In [3]:
%pip install -q langchain
%pip install -q langchain-community
%pip install -q langchain-text-splitters
%pip install -q langchain-google-genai
%pip install -q faiss-cpu
%pip install -q pypdf

**Step - 1 : Load the documents**

In [6]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader(
    "/content/GenAI",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()

print("Documents loaded:", len(documents))

Documents loaded: 28


**Step 2: Split documents into chunks**

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Chunks created:", len(chunks))

Chunks created: 90


**Step 3: Create Embeddings**

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Gemini Api Key
f=open(r"C:\Users\siva9\OneDrive\Desktop\GenAI_Project\Google_gemini_API_key.txt")
GEMINI_API_KEY = f.read()

# Create Embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=GEMINI_API_KEY
)

**Step 4: Create the FAISS vector database**

In [10]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

**Step 5: Implement retrieval**

In [11]:
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs={"k": 3}
)

**Step 6 : Initialize a Chat Prompt Template**

In [12]:
from langchain_core.prompts import ChatPromptTemplate

PROMPT_TEMPLATE = """
You are BankPrep AI, a banking exam preparation assistant.

Answer the question using only the supplied context.

Context:
{context}

Question:
{question}

Instructions:
- Explain the answer clearly in simple language.
- Use step-by-step explanations for calculations when relevant.
- Do not invent facts or information that is not supported by the context.
- If the answer is not available in the context, say:
  "I could not find this information in the provided study materials."
"""

prompt_template = ChatPromptTemplate.from_template(
    PROMPT_TEMPLATE
)

**Step - 7 : Initialize Gemini chat model and output parser**

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

chat_model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0.2
)

parser = StrOutputParser()

**Step 8 : Define a RAG Chain**

In [14]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt_template
    | chat_model
    | parser
)

**Invoke the chain**

In [16]:
query = "What is the IBPS Exam Pattern at present?"

answer = rag_chain.invoke(query)

print(answer)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided context, here is the exam pattern for IBPS PO and IBPS Clerk (Customer Service Associate):

---

### **1. IBPS PO Exam Pattern (2026 Revised)**

#### **A. Preliminary Exam (Qualifying Only)**
* **Structure:** 100 questions | 100 marks | 60 minutes total (20 minutes per section; switching between sections is not allowed).
* **Sectional Breakdown:**
  * **English Language:** 30 questions | 30 marks | 20 minutes
  * **Quantitative Aptitude:** 35 questions | 30 marks | 20 minutes
  * **Reasoning Ability:** 35 questions | 40 marks | 20 minutes
* *Note:* Prelims marks are not included in the final merit list. Candidates must clear IBPS cut-offs to reach Mains.

---

#### **B. Mains Exam**
* **Total Structure:** 225 total marks | 190 total minutes.

1. **Objective Test (170 questions | 200 marks | 160 minutes with separate sectional timing):**
   * **Reasoning:** 40 questions | 60 marks *(One source reports 45 minutes; revised notification is reported to have dropped Com